In [9]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from mlp import SimpleMLP

In [10]:
num_epochs = 100
lr = 0.01
batch_size = 64
hidden_num = 5
hidden_dim = 32
weight_decay = 1e-5

In [11]:
def train():
    # 读入处理后的数据
    print('\n================================== 读入处理后的数据 ==================================')
    train_data = pd.read_csv("./dataset/train_processed.csv")
    test_data = pd.read_csv("./dataset/test_processed.csv")
    
    train_x = train_data.drop(['Transported', 'PassengerId'], axis=1)
    train_y = train_data['Transported']
    test_x = test_data.drop(['Transported', 'PassengerId'], axis=1)
    
    # 将数据转换为 PyTorch 的 Tensor
    print('\n================================== 将数据转换为 PyTorch 的 Tensor ==================================')
    n_train = train_x.shape[0]
    n_test = test_x.shape[0]
    
    train_x = torch.tensor(train_x.values, dtype=torch.float32)
    train_y = torch.tensor(train_y.values, dtype=torch.float32).reshape(-1, 1)
    test_x = torch.tensor(test_x.values, dtype=torch.float32)
    print(train_x.shape, train_y.shape)
    
    # 构建 DataLoader
    train_ds = TensorDataset(train_x, train_y)
    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    
    # 定义模型、损失函数和优化器
    model = SimpleMLP(input_dim=train_x.shape[1], hidden_num=hidden_num, hidden_dim=hidden_dim, output_dim=1)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.BCELoss()
    
    # 训练模型
    print('\n================================== 训练模型 ==================================')
    for epoch in range(num_epochs):
        model.train()
        # 每个 epoch 的损失
        epoch_loss = 0
        
        # 预测正确的个数
        correct_num = 0
        for x_batch, y_batch in train_dl:
            y_pred = model(x_batch)
            correct_num += torch.sum((y_pred > 0.5) == y_batch).item() 
            loss = loss_fn(y_pred, y_batch)
            epoch_loss += loss.item()
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
        print(f"Epoch : {epoch}, Epoch Loss : {epoch_loss}, Accuracy : {correct_num / n_train}")
        
    # 预测测试集
    print('\n================================== 预测测试集 ==================================')
    model.eval()
    with torch.no_grad():
        y_pred = model(test_x)
        y_pred = (y_pred > 0.5).reshape(-1).cpu().numpy().astype(bool)
    sub = pd.DataFrame({'PassengerId': test_data['PassengerId'], 'Transported': y_pred})
    print(sub)
    sub.to_csv('submission.csv', index=False)
    
if __name__ == '__main__':
    train()
        


================================== 读入处理后的数据 ==================================

================================== 将数据转换为 PyTorch 的 Tensor ==================================
torch.Size([8693, 13]) torch.Size([8693, 1])

================================== 训练模型 ==================================
Epoch : 0, Epoch Loss : 96.08231270313263, Accuracy : 0.5082250086276314
Epoch : 1, Epoch Loss : 94.70655661821365, Accuracy : 0.5040837455423904
Epoch : 2, Epoch Loss : 93.95668596029282, Accuracy : 0.5051190613137007
Epoch : 3, Epoch Loss : 94.91555720567703, Accuracy : 0.5043138157137927
Epoch : 4, Epoch Loss : 94.29912620782852, Accuracy : 0.5024732543425745
Epoch : 5, Epoch Loss : 94.03551632165909, Accuracy : 0.5041987806280915
Epoch : 6, Epoch Loss : 93.9055073261261, Accuracy : 0.5007477280570574
Epoch : 7, Epoch Loss : 94.64623159170151, Accuracy : 0.49936730702864374
Epoch : 8, Epoch Loss : 93.79250919818878, Accuracy : 0.5071896928563212
Epoch : 9, Epoch Loss : 93.69711524248123, Accu

In [12]:
# import pandas as pd
# import numpy as np
# import torch
# import torch.nn as nn
# import torch.optim as optim
# from torch.utils.data import DataLoader, TensorDataset
# 
# from mlp import SimpleMLP
# 
# num_epochs = 100
# lr = 0.01
# batch_size = 64
# 
# hidden_dim = 2
# hidden_num = 2
# weight_decay = 1e-5
# 
# def main():
#     # 读入处理后的数据
#     print('\n================================== 读入处理后的数据 ==================================')
#     df_train = pd.read_csv('dataset/train_processed.csv')
#     df_test = pd.read_csv('dataset/test_processed.csv')
#     
#     df_train_features = df_train.drop(['Transported', 'PassengerId'], axis=1)
#     df_train_target = df_train['Transported']
#     df_test_features = df_test.drop(['Transported', 'PassengerId'], axis=1)
#     print(df_train_features)
#     print(df_train_target)
#     
#     # 将数据转换为 PyTorch 的 Tensor
#     print('\n================================== 将数据转换为 PyTorch 的 Tensor ==================================')
#     n_train = df_train.shape[0]
#     n_test = df_test.shape[0]
#     
#     X_train = torch.tensor(df_train_features.values, dtype=torch.float32)
#     y_train = torch.tensor(df_train_target.values, dtype=torch.float32).reshape(-1, 1)
#     X_test = torch.tensor(df_test_features.values, dtype=torch.float32)
#     print(X_train.shape)
#     print(y_train.shape)
#     
#     # 构建 DataLoader
#     train_dataset = TensorDataset(X_train, y_train)
#     train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
#     
#     # 定义模型、损失函数和优化器
#     model = SimpleMLP(input_dim=X_train.shape[1], hidden_num=hidden_num, hidden_dim=hidden_dim, output_dim=1)
#     optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
# 
#     # 二元交叉熵损失函数(Binary Cross Entropy)，用于二分类任务
#     loss = nn.BCELoss()
#     
#     # 训练模型
#     print('\n================================== 训练模型 ==================================')
#     for epoch in range(num_epochs):
#         model.train()
#         
#         # 每个 epoch 的损失
#         epoch_loss = 0
#         
#         # 预测正确的个数
#         correct_num = 0
#         for X_batch, y_batch in train_loader:
#             y_pred = model(X_batch)
#             correct_num += torch.sum((y_pred > 0.5) == y_batch).item()
#             
#             l = loss(y_pred, y_batch)
#             epoch_loss += l.item()
# 
#             optimizer.zero_grad()
#             l.backward()
#             optimizer.step()
#         
#         print(f'Epoch: {epoch}, Epoch Loss: {epoch_loss}, Accuracy: {correct_num / n_train}')
#     
#     # 预测测试集
#     print('\n================================== 预测测试集 ==================================')
#     model.eval()
#     with torch.no_grad():
#         y_pred = model(X_test)
#         y_pred = (y_pred > 0.5).reshape(-1).cpu().numpy().astype(bool)
#     sub = pd.DataFrame({'PassengerId': df_test['PassengerId'], 'Transported': y_pred})
#     print(sub)
#     sub.to_csv('submission.csv', index=False)
# 
# if __name__ == '__main__':
#     main()



================================== 读入处理后的数据 ==================================
      HomePlanet  CryoSleep  Destination       Age  VIP  RoomService  \
0              1          0            2  0.710909    0    -0.344192   
1              0          0            2 -0.331680    0    -0.175878   
2              1          0            2  2.031522    1    -0.277793   
3              1          0            2  0.293873    0    -0.344192   
4              0          0            2 -0.887728    0     0.123691   
...          ...        ...          ...       ...  ...          ...   
8688           1          0            0  0.849921    1    -0.344192   
8689           0          1            1 -0.748716    0    -0.344192   
8690           0          0            2 -0.192668    0    -0.344192   
8691           1          0            0  0.224367    0    -0.344192   
8692           1          0            2  1.058439    0    -0.149627   

      FoodCourt  ShoppingMall       Spa    VRDeck  Deck